# Session 1 · Part 2 — Quality control and feature summaries

**Independent checkpoint:** reload the data, calculate QC summaries, and save both tables and a diagnostic figure.


In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
for candidate in (current, *current.parents):
    if (candidate / "src" / "dgat_tutorial").is_dir():
        tutorial_root = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter inside the hands-on_tutorial directory.")

sys.path.insert(0, str(tutorial_root / "src"))

from dgat_tutorial.checkpoints import tutorial_paths, write_checkpoint

paths = tutorial_paths(tutorial_root)
print(f"Tutorial root: {paths.root}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dgat_tutorial.data import load_tutorial_data

dataset = load_tutorial_data(paths.raw_data, allow_demo=True)
transcripts = dataset.transcripts.select_dtypes(include=[np.number])
proteins = dataset.proteins.select_dtypes(include=[np.number])

transcript_library_size = transcripts.sum(axis=1)
detected_genes = (transcripts > 0).sum(axis=1)
protein_total = proteins.sum(axis=1)


In [ ]:
def summarize_features(matrix, feature_type, n=10):
    table = pd.DataFrame({
        "feature": matrix.columns,
        "feature_type": feature_type,
        "mean": matrix.mean(axis=0).to_numpy(),
        "variance": matrix.var(axis=0).to_numpy(),
        "nonzero_fraction": (matrix > 0).mean(axis=0).to_numpy(),
    })
    return table.sort_values("variance", ascending=False).head(n)

top_transcripts = summarize_features(transcripts, "transcript")
top_proteins = summarize_features(proteins, "protein")
top_transcripts


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].hist(transcript_library_size, bins=30, color="#2f2f2f")
axes[0].set_title("Transcript library size")
axes[1].hist(detected_genes, bins=30, color="#5b7f95")
axes[1].set_title("Detected genes per spot")
axes[2].hist(protein_total, bins=30, color="#b45a3c")
axes[2].set_title("Protein signal per spot")
plt.tight_layout()

transcript_path = paths.results / "session01_top_transcripts.csv"
protein_path = paths.results / "session01_top_proteins.csv"
figure_path = paths.figures / "session01_basic_statistics.png"
top_transcripts.to_csv(transcript_path, index=False)
top_proteins.to_csv(protein_path, index=False)
plt.savefig(figure_path, dpi=160)
plt.show()

manifest = write_checkpoint(
    "1.2", [transcript_path, protein_path, figure_path],
    summary={"spots": len(transcripts)}, start=paths.root
)
print(f"Checkpoint written: {manifest}")


## Checkpoint

Use the saved feature tables to identify strong, variable markers before spatial plotting.